<a href="https://colab.research.google.com/github/suyon0416/26-2_ESAA_YB/blob/main/YB_1106_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C_%EB%B6%84%EB%A5%982.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Lightgbm

* 데이터 '와인품질분류.zip'입니다.

1. 필요한 라이브러리를 가져옵니다.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import lightgbm as lgbm

2. 데이터를 불러옵니다.

In [3]:
train = pd.read_csv('/content/drive/MyDrive/ESAA data/wine_data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/ESAA data/wine_data/test.csv')
smpl_sub = pd.read_csv('/content/drive/MyDrive/ESAA data/wine_data/sample_submission.csv')

In [4]:
train.head()

,index,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,white
1,1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,red
2,2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,white
3,3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,white
4,4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,white


In [5]:
test.head()

,index,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,white
1,1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,red
2,2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,white
3,3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,white
4,4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,white


3. 변수를 변환합니다.

In [9]:
# type에는 white와 red 두 종류가 있습니다.
# 각각 0,1로 변환합니다.
train['type'] = train['type'].map({'red': 0, 'white': 1})
test['type'] = test['type'].map({'red': 0, 'white': 1})

In [10]:
train.head()

,index,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,1
1,1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,0
2,2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,1
3,3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,1
4,4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,1


In [11]:
test.head()

,index,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,1
1,1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,0
2,2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,1
3,3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,1
4,4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,1


In [12]:
# 모델에 입력하기 전 데이터를 정형합니다.
# train_x : train의 'index', 'quality' 피처를 제거하고 저장합니다.
# train_y : train의 'quality'를 저장합니다.
# test_x : test의 'index'를 제거하고 저장합니다.

train_x = train.drop(columns=['index', 'quality'])
train_y = train['quality']
test_x = test.drop(columns=['index'])

In [13]:
train_x.shape, train_y.shape, test_x.shape

((5497, 12), (5497,), (1000, 12))

In [14]:
train_x.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,1
1,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,0
2,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,1
3,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,1
4,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,1


In [15]:
train_y.head()

,quality
0,5
1,5
2,5
3,6
4,6


In [16]:
test_x.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,1
1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,0
2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,1
3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,1
4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,1


4. LightGBM 모델을 생성 및 훈련합니다.

In [17]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 학습 데이터와 검증 데이터 분리
X_tr, X_val, y_tr, y_val = train_test_split(
    train_x, train_y,
    test_size=0.2,
    random_state=42,
    stratify=train_y
)

# LightGBM 모델 생성
lgbm_clf = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
)

# 모델 학습
lgbm_clf.fit(X_tr, y_tr)

# 검증 데이터 예측
val_pred = lgbm_clf.predict(X_val)

# 정확도 확인
print("검증 정확도:", accuracy_score(y_val, val_pred))

# 전체 학습 데이터로 최종 학습
lgbm_clf.fit(train_x, train_y)


검증 정확도: 0.63


LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=42,
               verbosity=-1)

5. 생성한 모델로 예측 데이터를 생성합니다.

In [18]:
# test 데이터 예측
pred = lgbm_clf.predict(test_x)

# 예측 결과 확인
print("예측 데이터 개수:", len(pred))
print("예측된 품질 등급 분포:")
print(pd.Series(pred).value_counts().sort_index())

예측 데이터 개수: 1000
예측된 품질 등급 분포:
4     10
5    366
6    481
7    131
8     12
Name: count, dtype: int64


6. 제출파일을 생성합니다.

In [23]:
# 제출 양식 복사
smpl_sub1 = smpl_sub.copy()

# 제출 파일 확인
display(smpl_sub1.head())

,index,quality
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [24]:
#서브미션 파일의 quality 변수에 예측값을 대입합니다.
smpl_sub1['quality'] = pred

In [25]:
display(smpl_sub1.head())

# 제출 데이터 확인
print("제출 데이터 크기:", smpl_sub1.shape)
print("예측값 개수:", len(pred))
print("결측치 개수:", smpl_sub1.isnull().sum().sum())

,index,quality
0,0,5
1,1,5
2,2,6
3,3,5
4,4,6


제출 데이터 크기: (1000, 2)
예측값 개수: 1000
결측치 개수: 0


In [26]:
# csv 파일로 저장합니다.
smpl_sub.to_csv('submission.csv', index=False)

# 2. XGBoost

In [27]:
# 데이터 다운로드 링크로 데이터를 코랩에 불러옵니다.

!wget 'https://bit.ly/3i4n1QB'

import zipfile
with zipfile.ZipFile('3i4n1QB', 'r') as existing_zip:
    existing_zip.extractall('data')

--2026-11-06 10:09:00--  https://bit.ly/3i4n1QB
Resolving bit.ly (bit.ly)... 67.199.248.10, 67.199.248.11
Connecting to bit.ly (bit.ly)|67.199.248.10|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://drive.google.com/uc?export=download&id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw [following]
--2026-11-06 10:09:00--  https://drive.google.com/uc?export=download&id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw
Resolving drive.google.com (drive.google.com)... 74.125.203.101, 74.125.203.100, 74.125.203.138, ...
Connecting to drive.google.com (drive.google.com)|74.125.203.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw&export=download [following]
--2026-11-06 10:09:00--  https://drive.usercontent.google.com/download?id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 64.

In [28]:
# 라이브러리 및 데이터 불러오기
import pandas as pd
from xgboost import XGBClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.drop('index',axis = 1 ,inplace =True)
test.drop('index',axis = 1 ,inplace =True)

In [29]:
# 데이터 확인
print("Train 데이터 크기:", train.shape)
print("Test 데이터 크기:", test.shape)

display(train.head())
display(test.head())

print("와인 품질 분포:")
print(train['quality'].value_counts().sort_index())

Train 데이터 크기: (5497, 13)
Test 데이터 크기: (1000, 12)


,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,white
1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,red
2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,white
3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,white
4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,white


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type
0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,white
1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,red
2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,white
3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,white
4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,white


와인 품질 분포:
quality
3      26
4     186
5    1788
6    2416
7     924
8     152
9       5
Name: count, dtype: int64


In [30]:
# 원핫 인코딩 (pd.get_dummies())

train = pd.get_dummies(train, columns=['type'], drop_first=True, dtype=int)
test = pd.get_dummies(test, columns=['type'], drop_first=True, dtype=int)

display(train.head())
display(test.head())

,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type_white
0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,1
1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,0
2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,1
3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,1
4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,1


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type_white
0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,1
1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,0
2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,1
3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,1
4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,1


In [31]:
# 모델 정의
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

In [32]:
# 모델 학습
# X 는 train에서 quality 를 제외한 모든 변수
# y 는 train의 qulity 변수

X = train.drop(columns=['quality'])
y = train['quality']

# 품질 등급을 0부터 시작하는 정수로 변환
y_encoded = label_encoder.fit_transform(y)

# fit 메소드를 이용해 모델 학습
model.fit(X, y_encoded)

print("XGBoost 모델 학습 완료!")


XGBoost 모델 학습 완료!


In [33]:
# predict 메소드와 원핫 인코딩한 test 데이터를 이용해 품질 예측

# 학습 데이터와 test 데이터의 컬럼 구성 일치
test_encoded = test.reindex(columns=X.columns, fill_value=0)

# predict 메소드로 품질 예측
pred_encoded = model.predict(test_encoded)

# 원래 와인 품질 등급으로 복원
pred = label_encoder.inverse_transform(pred_encoded)

print("예측 결과:")
print(pd.Series(pred).value_counts().sort_index())

예측 결과:
4      9
5    368
6    484
7    125
8     14
Name: count, dtype: int64


In [34]:
# sample_submission.csv 파일을 불러와 예측된 값으로 채워 주기
# 제출 양식 불러오기
smpl_sub = pd.read_csv('data/sample_submission.csv')

# 예측값 대입
smpl_sub['quality'] = pred

# 제출 파일 저장
smpl_sub.to_csv('xgboost_submission.csv', index=False)

display(smpl_sub.head())

print("제출 파일 생성 완료!")

,index,quality
0,0,5
1,1,5
2,2,5
3,3,5
4,4,7


제출 파일 생성 완료!


In [36]:
# 정답파일 내보내기
smpl_sub.to_csv('xgb_pred.csv',index = False)

# 3. 보팅

In [37]:
# 데이터 다운로드 링크로 데이터를 코랩에 불러옵니다.

!wget 'https://bit.ly/3i4n1QB'

import zipfile
with zipfile.ZipFile('3i4n1QB', 'r') as existing_zip:
    existing_zip.extractall('data')

--2026-11-06 10:13:31--  https://bit.ly/3i4n1QB
Resolving bit.ly (bit.ly)... 67.199.248.10, 67.199.248.11
Connecting to bit.ly (bit.ly)|67.199.248.10|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://drive.google.com/uc?export=download&id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw [following]
--2026-11-06 10:13:31--  https://drive.google.com/uc?export=download&id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw
Resolving drive.google.com (drive.google.com)... 64.233.188.100, 64.233.188.139, 64.233.188.113, ...
Connecting to drive.google.com (drive.google.com)|64.233.188.100|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw&export=download [following]
--2026-11-06 10:13:32--  https://drive.usercontent.google.com/download?id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 64.

In [38]:
# 라이브러리 및 데이터 불러오기
import pandas as pd
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.drop('index',axis = 1 ,inplace =True)
test.drop('index',axis = 1 ,inplace =True)

In [42]:
# 데이터 확인

print("Train 데이터 크기:", train.shape)
print("Test 데이터 크기:", test.shape)

display(train.head())
display(test.head())

print("결측치 확인")
print(train.isnull().sum())

Train 데이터 크기: (5497, 13)
Test 데이터 크기: (1000, 12)


,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type_white
0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,1
1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,0
2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,1
3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,1
4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,1


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,type_white
0,9.0,0.31,0.48,6.6,0.043,11.0,73.0,0.99380,2.90,0.38,11.6,1
1,13.3,0.43,0.58,1.9,0.070,15.0,40.0,1.00040,3.06,0.49,9.0,0
2,6.5,0.28,0.27,5.2,0.040,44.0,179.0,0.99480,3.19,0.69,9.4,1
3,7.2,0.15,0.39,1.8,0.043,21.0,159.0,0.99480,3.52,0.47,10.0,1
4,6.8,0.26,0.26,2.0,0.019,23.5,72.0,0.99041,3.16,0.47,11.8,1


결측치 확인
quality                 0
fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
type_white              0
dtype: int64


In [44]:
# 원핫 인코딩 (pd.get_dummies())

train = pd.get_dummies(train, columns=['type'], drop_first=True, dtype=int)
test = pd.get_dummies(test, columns=['type'], drop_first=True, dtype=int)

display(train.head())
display(test.head())

KeyError: "None of [Index(['type'], dtype='object')] are in the [columns]"

In [45]:
# 모델 정의 (LightGBM, XGBoost, Random Forest)
lgbm_clf = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    verbosity=-1
)

xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    eval_metric='mlogloss',
    random_state=42
)

rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# VotingClassifier 정의
voting_clf = VotingClassifier(
    estimators=[
        ('lgbm', lgbm_clf),
        ('xgb', xgb_clf),
        ('rf', rf_clf)
    ],
    voting='soft',
    n_jobs=-1
)



In [46]:
# 모델 학습
# X는 train에서 quality를 제외한 모든 변수
# y는 train의 quality 변수

X = train.drop(columns=['quality'])
y = train['quality']

# XGBoost 사용을 위해 레이블 인코딩
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 학습 데이터와 테스트 데이터 컬럼 일치
test = test.reindex(columns=X.columns, fill_value=0)

# fit 메소드를 이용해 모델 학습
voting_clf.fit(X, y_encoded)

print("Voting 앙상블 모델 학습 완료!")


Voting 앙상블 모델 학습 완료!


In [47]:
# predict 메소드와 원핫 인코딩한 테스트 데이터를 이용해 품질 예측

pred_encoded = voting_clf.predict(test)

# 원래 와인 품질 등급으로 복원
pred = label_encoder.inverse_transform(pred_encoded)

print("예측 품질 분포")
print(pd.Series(pred).value_counts().sort_index())

예측 품질 분포
4      8
5    360
6    494
7    126
8     12
Name: count, dtype: int64
